In [0]:
%pip install topojson
dbutils.library.restartPython()

This notebook demonstrates how to read a geojson from a volume, convert it to a geospatial dataframe, and upload it as a geospatial table in databricks. From there, Carto can read the geospatial table, no conversion necessary.

In [0]:
from pyspark.sql.functions import expr, col

# Read GeoJSON file from volume
# Replace with your actual volume path
topojson_path = "/Volumes/scorecard_fulcrum/geo/scoreacard_geospatial_files/2020NTAMap.json"
# Read the GeoJSON file
df = spark.read.option("multiline", "true").json(topojson_path)

# Display the schema to understand the structure
df.printSchema()
display(df.limit(5))

Parse the topojson into geometry

This is vibe coded to hell and back to fit the schmea of this particular nta json.

also vibecoded


In [0]:
import json
import topojson as tp
from pyspark.sql.types import StructType, StructField, StringType, BinaryType

# Read the TopoJSON file as text
with open("/Volumes/scorecard_fulcrum/geo/scoreacard_geospatial_files/2020NTAMap.json", "r") as f:
    topojson_data = json.load(f)

print(f"Objects in TopoJSON: {list(topojson_data.get('objects', {}).keys())}")

# Convert TopoJSON to GeoJSON using topojson library
topo = tp.Topology(topojson_data)
geojson_features = []

# Extract features from all objects in the TopoJSON
for obj_name in topojson_data.get('objects', {}).keys():
    print(f"Processing object: {obj_name}")
    # Convert each object to GeoJSON
    geojson = topo.to_geojson(obj_name)
    
    print(f"GeoJSON result for {obj_name}: {type(geojson)}")
    
    # Skip if conversion returned None
    if geojson is None:
        print(f"Skipping {obj_name} - conversion returned None")
        continue
    
    # Extract features
    if geojson.get('type') == 'FeatureCollection':
        geojson_features.extend(geojson.get('features', []))
    elif geojson.get('type') == 'Feature':
        geojson_features.append(geojson)

print(f"Extracted {len(geojson_features)} features from TopoJSON")

# Check if we have any features before creating DataFrame
if len(geojson_features) == 0:
    raise ValueError("No features extracted from TopoJSON. Check the object names and structure.")

# Convert to Spark DataFrame
features_json = [json.dumps(feature) for feature in geojson_features]
features_df = spark.createDataFrame([(f,) for f in features_json], ["feature_json"])

# Parse GeoJSON features and convert to WKB geometry
gdf = features_df.select(
    expr("from_json(feature_json, 'struct<type:string,properties:map<string,string>,geometry:struct<type:string,coordinates:array<array<array<array<double>>>>>>') as feature")
).select(
    expr("feature.properties.*"),
    expr("ST_AsBinary(ST_GeomFromGeoJSON(to_json(feature.geometry)))").alias("geometry")
)

# Display the result
display(gdf.limit(5))

In [0]:
%sql
--DROP TABLE IF EXISTS scorecard_fulcrum.carto.nta_geospatial_table

In [0]:
# Define your target table name
target_table = "scorecard_fulcrum.carto.nta_geospatial_table"

# Write to Delta table
gdf.write.mode("overwrite").saveAsTable(target_table)

print(f"Geospatial table created: {target_table}")
print(f"Total rows: {gdf.count()}")

In [0]:
gdf = spark.read()